# 节点 2：跑完预训练、用 KMeans 初始化好簇心、并完成 ML/CL 索引池的抽样，正要进入 400 轮聚类迭代的 for epoch 循环（DSSC.py:262）

In [ ]:
# 获取先前保存的变量
import scanpy as sc
import numpy as np
import scipy.sparse as sp

adata = sc.read_h5ad("./data/adata.h5ad")

A   = sp.load_npz("./data/A.npz")
A_n = sp.load_npz("./data/A_n.npz")

d = np.load("./data/arrays.npz", allow_pickle=True)  # 含 p_ 时要 allow_pickle
count_X = d["count_X"]
pos_ = d["pos_"]
y = d["y"]
ml_ind1 = d["ml_ind1"]; ml_ind2 = d["ml_ind2"]
cl_ind1 = d["cl_ind1"]; cl_ind2 = d["cl_ind2"]
n_clusters = int(d["n_clusters"])

/home/liuzy/Software/miniconda3/envs/dssc-4090/lib/python3.8/site-packages/setuptools_scm/_integration/setuptools.py:31: RuntimeWarning: 
ERROR: setuptools==59.8.0 is used in combination with setuptools_scm>=8.x

Your build configuration is incomplete and previously worked by accident!
setuptools_scm requires setuptools>=61

Suggested workaround if applicable:
 - migrating from the deprecated setup_requires mechanism to pep517/518
   and using a pyproject.toml to declare build dependencies
   which are reliably pre-installed before running the build tools

  warnings.warn(


In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path("../../src").resolve()))
from DSSC import DSSC

# 创建模型对象
model = DSSC(input_dim=adata.n_vars, encodeLayer=[128], decodeLayer=[128], encodeHead=3, 
            encodeConcat=False, gamma=0.1, activation="elu", z_dim=32, sigma = 0.1, device="cuda").to("cuda")

## 1. 预训练 train_model（run_DSSC.py:147 → DSSC.py:148-178）

### 进入该模块的数据：
| 数据 | 来源（节点 1 处） | 变量名 | 位置                                            |
| --- | ---------------- | ----- | ---------------------------------------------- |
| adata.X (4221×2000, log 归一化) | h5 X → HVG | X_v | DSSC.py:149,155 |
| count_X (4221×2000, 原始 counts) | h5 X → 同 HVG 列 | X_raw_v | DSSC.py:152,158 |
| size_factors (4221,) | h5 X 行和 | size_factor_v | DSSC.py:153,159 |
| A_n (4221×4221, 归一化邻接) | h5 Pos → kNN=20 → norm_adj | dgl 图 G | DSSC.py:150 |
| A (dense 邻接) | 同上 | A_v（预训练里未使用） | DSSC.py:157 |
| ml / cl 索引 | — | 本阶段完全不参与 | — |

In [ ]:
from time import time

print(str(model))
t0 = time()
model.train_model(adata.X, A_n, A, count_X, adata.obs.size_factors, lr=0.001, train_iter=200, verbose=True)
print('Pre-training time: %d seconds.' % int(time() - t0))

DSSC(
  (encoder): GATEncoder(
    (layers): ModuleList(
      (0): GATConv(
        (fc): Linear(in_features=2000, out_features=384, bias=False)
        (feat_drop): Dropout(p=0.0, inplace=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (leaky_relu): LeakyReLU(negative_slope=0.2)
      )
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ELU(alpha=1.0)
    )
    (enc_mu): GATConv(
      (fc): Linear(in_features=128, out_features=96, bias=False)
      (feat_drop): Dropout(p=0, inplace=False)
      (attn_drop): Dropout(p=0, inplace=False)
      (leaky_relu): LeakyReLU(negative_slope=0.2)
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
  )
  (_dec_mean): Sequential(
    (0): Linear(in_features=128, out_features=2000, bias=True)
    (1): MeanAct()
  )
  (_dec_d

### 一次前向的数据流（DSSC.py:165 → aeForward DSSC.py:112-121）
1. 加噪：x_c = X_v + 0.1·randn（DSSC.py:113，sigma=0.1），构成去噪自编码器输入。
2. GAT 编码器（DSSC.py:114）：z = encoder(G, x_c)
- 由于 encodeConcat 未被传入（节点 1 记录的问题），实际走 concat=False 分支（DSSC.py:70-75）：
    - GATConv(2000→128, 3 heads)：沿 A_n 把每个细胞与它的 20 个空间邻居做图注意力聚合，输出 (4221,3,128)；
    - torch.sum(dim=1) 合并 3 个头 → (4221,128)；随后 BatchNorm(128) 被连续调用两次、ELU 被跳过（DSSC.py:74-75）；
- enc_mu = GATConv(128→32, 3 heads) → sum heads → z0/z (4221,32)。
- 关键点：这里的图卷积使隐表示融合了空间邻域信息，而不是逐细胞独立编码。
3. 解码器（DSSC.py:115）：x_ = decoder(z)，即 Linear(32→128)+BN+ELU（DSSC.py:94）→ (4221,128)。
4. 三个输出头（DSSC.py:95-97,116-118）：
- mean = exp(clamp(Linear(128→2000)))（MeanAct，layers.py:239）
- disp = softplus(clamp(Linear(128→2000)))（DispAct，layers.py:247）
- pi = sigmoid(Linear(128→2000))（零膨胀概率）
- 每个形状 (4221,2000)。
5. 返回 z0, mean, disp, pi（DSSC.py:121）；注意第 120 行又用干净输入 X_v 跑了一遍编码器，得到 z0 供后续聚类用（多一次前向）。

### 损失与反传（DSSC.py:166-170）
- loss_zinb = ZINBLoss(x=X_raw_v, mean, disp, pi, scale_factor=size_factor_v)（layers.py:299-318）：
    - mean = mean * scale_factor（把表达均值还原到 count 尺度）；
    - 对原始 counts 计算零膨胀负二项负对数似然，x≈0 用 zero_case、否则 nb_case，最后 mean 归约。
- loss.backward() + optim.Adam(amsgrad=True).step()。
- 重复 train_iter=200 次；每次还额外做一次 eval 前向仅用于打印（DSSC.py:173-178，日志里的 Iteration:i, ZINB loss:...）。
到这里 adata.X/count_X/size_factors/A_n 的使命：把 counts 的 ZINB 结构（尤其零膨胀）压进 32 维空间隐表示 z，并让编码器学到空间图上的表示。

## 2. 进入聚类 fit（run_DSSC.py:151 → DSSC.py:204-248）

### 重新装载 + KMeans 初始化（DSSC.py:209-233）

- 再次 torch.tensor(adata.X)、重建 dgl 图 G（DSSC.py:209-211）。
- self.mu = Parameter(7, 32)（DSSC.py:221，n_clusters=7 来自断点 #1 的 Y）。
- Zdata = encodeBatch(X, A_n)（DSSC.py:225）→ 用干净 X 过编码器得到 (4221,32) 隐表示。
- KMeans(7, n_init=20).fit_predict(Zdata)（DSSC.py:224-226）；簇心写入 self.mu（DSSC.py:228）。
- y（金标准）在此只用于打印初始 ACC/NMI/ARI（DSSC.py:230-233），不参与梯度。

### ML/CL 索引池抽样（DSSC.py:236-248）

- ml_pool1/2 = ml_ind1/2（26203 对），cl_pool1/2 = cl_ind1/2（31828 对）——就是断点 #1 从两个 txt 读入、减 1 后的数组。
- 从各自池中随机无放回抽 n_ml=4000、n_cl=4000 对（DSSC.py:240-247），打印 Constraints summary: ML/CL=4000。
- 计算 batch 数备用（DSSC.py:250-255）。

此处 ml/cl 才第一次真正“接入”模型，但它们仍只是索引；真正作用在 soft_assign 产生的软分配 q 上要等聚类循环里（DSSC.py:313-315 / 328-330）。

In [ ]:
y_pred, final_loss, epoch = model.fit(X=adata.X, X_raw = count_X, X_sf=adata.obs.size_factors, A = A, A_n = A_n,
        n_clusters = n_clusters, num_epochs=args.clustering_iters, y=y, n_ml = args.n_ml, n_cl = args.n_cl,
        ml_ind1=ml_ind1, ml_ind2=ml_ind2, cl_ind1=cl_ind1, cl_ind2=cl_ind2, ml_p=args.weight_ml, cl_p=args.weight_cl,
        p_=p_, lr = args.lr, update_interval=1, tol=0.001)